The following exercises are meant to be solved by gathering the bash commands incrementally in two scripts, one for ex 1.* the other for ex 2.* 

### Ex 1

1\.a Make a new directory called `students` in your home. Download a csv file with the list of students of this lab from [here](https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv) (use the `wget` command) and copy that to `students`. First check whether the file is already there

```
if [ ! -d "$HOME/students" ]; then mkdir "$HOME/students"
fi

wget https://www.dropbox.com/scl/fi/bxv17nrbrl83vw6qrkiu9/LCP_22-23_students.csv

if [ ! -f "$HOME/students/LCP_22-23_students.csv" ]; then
    wget -O "$HOME/students/LCP_22-23_students.csv" https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv
else
    echo "File already exists in the students directory."
fi
```

1\.b Make two new files, one containing the students belonging to PoD, the other to Physics.

```
input_file="$HOME/students/LCP_22-23_students.csv"
pod_file="$HOME/students/PoD_students.csv"
physics_file="$HOME/students/Physics_students.csv"

if [ ! -f "$input_file" ]; then
    echo "Input file $input_file not found!"
    exit 1
fi

echo "Visualizing the general students file:"
echo ""
head "$input_file"

echo "Filtering the students into:"
echo "- $pod_file"
echo "- $physics_file""
grep "PoD" "$input_file" > "$pod_file"
grep "Physics" "$input_file" > "$physics_file"
echo "Visualizing PoD students:"
head "$pod_file"
echo "Visualizing Physics students:"
head "$physics_file"
```

1\.c For each letter of the alphabet, count the number of students whose surname starts with that letter.

```
input_file="$HOME/students/LCP_22-23_students.csv"

echo "Counting students' surnames by the first letter..."
cut -d ',' -f 1 "$input_file" |
    sed '1d' |
    awk '{print substr($1,1,1)}' |
    sort |
    uniq -c |
    awk '{printf "%s: %s\n", $2, $1}'
```

1\.d Find out which is the letter with most counts.

```
input_file="$HOME/students/LCP_22-23_students.csv"

cut -d ',' -f 1 "$input_file" |
    sed '1d' |
    awk '{print substr($1,1,1)}' |
    uniq -c |
    sort -nr |
    head -n 1
```

1\.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file

```
for i in $(seq 1 18); do 
    group_file="$HOME/students/group_$i.csv" 
    > "$group_file"
    data=$(tail -n +2 "$HOME/students/LCP_22-23_students.csv")
    echo "Creating Group $i:" 
    for j in $(seq $i 18 72); do
        echo "$j"
        echo "$group_file"
        echo "$data" | sed -n "${j}p" >> "$group_file"
    done
    echo "Group $i:"
    cat "$group_file"
    echo "--------------------------"
done
```

### Ex 2

2.a Make a copy of the file `data.csv` removing the metadata and the commas between numbers; call it `data.txt`

```
tail -n +5 data.csv | tr -d ',' > data.txt
```

2\.b How many even numbers are there?

```
grep -o '[0-9]\+' data.txt | 
awk '$1 % 2 == 0 {count++} END {print count}'
```

2\.c Distinguish the entries on the basis of `sqrt(X^2 + Y^2 + Z^2)` is greater or smaller than `100*sqrt(3)/2`. Count the entries of each of the two groups 

```
input_file="data.txt"
threshold=$(echo "100 * sqrt(3) / 2" | bc -l)

greater_count=0
smaller_count=0

while IFS=' ' read -r X Y Z rest; do
    magnitude=$(echo "sqrt($X^2 + $Y^2 + $Z^2)" | bc -l)

    if (( $(echo "$magnitude > $threshold" | bc -l) )); then
        ((greater_count++))
    else
        ((smaller_count++))
    fi
done < "$input_file"

echo "Entries with value > 100*sqrt(3)/2 - Count: $greater_count"
echo "Entries with value <= 100*sqrt(3)/2 - Count: $smaller_count"
```

2\.d Make `n` copies of data.txt (with `n` an input parameter of the script), where the i-th copy has all the numbers divided by i (with `1<=i<=n`).

```
n=$1
input_file="data.txt"
for ((i=1; i<=n; i++)); do
    output_file="data_$i.txt"
    echo "Creating $output_file"
    awk -v divisor=$i '{for (j=1; j<=NF; j++) $j=$j/divisor}1' "$input_file" > "$output_file"
    echo "Created $output_file"
done
```